In [1]:
# === IMPORT LIBRARIES ===
import os
import sys
import subprocess
import ctypes

# === CUDA SYSTEM BOOT FIX ===

# Force-inject CUDA library to RAM before ANY module imports!
try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
except Exception:
    pass

# SYSTEM HOTFIX: Inject absolute path to CUDA 13.0 linker libraries
# This guarantees that bitsandbytes and 4-bit quantization load flawlessly on this server!
cuda_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib"
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + cuda_link_path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
# === FILEPATH SETUP ===

# 1. Inject Codebase into Python Path (Wipe cache first for Jupyter safety)
import sys
for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)
print(f"✅ Codebase mounted at: {codebase_path}")

# 2. Configure Global Filepaths
CACHE_DIR = f"{codebase_path}/output/cache"
ENV_PATH = f"{codebase_path}/artifacts/.env"
MODELS_DIR = f"{codebase_path}/output/models"

✅ Codebase mounted at: ../


In [3]:
# === IMPORT HF API KEY ===
from huggingface_hub import login

# Load HF token from artifacts/.env
env_path = ENV_PATH
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        for line in f:
            if line.strip() and not line.startswith("#") and "=" in line:
                key, val = line.strip().split("=", 1)
                if key.strip() == "HF_TOKEN":
                    login(token=val.strip())
                    print("Successfully logged into Hugging Face Hub!")
                    break
else:
    print(f"Warning: {env_path} not found.")

Successfully logged into Hugging Face Hub!


In [4]:
# === CONFIGURATION ===
USE_UNSLOTH = False

MODEL_ID = "mistralai/Mistral-Nemo-Instruct-2407"

STRUCTONLY_OUTPUT_DIR = f"{MODELS_DIR}/sft_Mistral-Nemo-Instruct-2407_structOnly_LoRA"
FULLINFO_OUTPUT_DIR = f"{MODELS_DIR}/sft_Mistral-Nemo-Instruct-2407_fullInfo_LoRA"

In [5]:
# === IMPORT LIBRARIES ===
import torch
# import sentencepiece
# import tiktoken
from datasets import load_dataset
from transformers import (
	AutoModelForCausalLM,
	AutoTokenizer,
	BitsAndBytesConfig,
	EarlyStoppingCallback,
)

if USE_UNSLOTH:
	# from ._gpu_init import *
	from unsloth import FastLanguageModel
else:
	from transformers.training_args import TrainingArguments
	from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
	
from trl import SFTConfig, SFTTrainer
from src.utils.prompts import format_prompt

In [6]:
# === LOAD TOKENIZER & PREPARE SCHEMAS ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, 
                                          trust_remote_code=True, 
                                          # use_fast=False,
                                          )
tokenizer.pad_token = tokenizer.eos_token

# Compute data type: bfloat16 for local server
COMPUTE_DTYPE = torch.bfloat16
print(f"Targeting compute data type: {COMPUTE_DTYPE}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

# Apply formatting
import json
def apply_chat_template(example, tokenizer):
    example = json.loads(example['text'])
    messages = format_prompt(example)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': prompt}

[transformers] The tokenizer you are loading from 'mistralai/Mistral-Nemo-Instruct-2407' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Targeting compute data type: torch.bfloat16


In [7]:
# === LoRA CONFIG ===
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    # === TRYING OTHER SFT MTHODS (DoRA, PiSSA) ===
    # use_dora=True,
    # init_lora_weights="pissa"  # or just "pissa_niter_16"
)

In [8]:
# === MULTI-CONFIG SFT TRAINING LOOP ===
import gc
from transformers.trainer_utils import get_last_checkpoint

max_seq_length = 1500

configs = [
	{
		"prompt_format": "structOnly",
		"train_path": f"{CACHE_DIR}/train_structural.jsonl",
		"val_path": f"{CACHE_DIR}/val_structural.jsonl",
		"output_dir": STRUCTONLY_OUTPUT_DIR
	},
	{
		"prompt_format": "fullInfo",
		"train_path": f"{CACHE_DIR}/train_full_info.jsonl",
		"val_path": f"{CACHE_DIR}/val_full_info.jsonl",
		"output_dir": FULLINFO_OUTPUT_DIR
	}
]

for config in configs:
	format_name = config["prompt_format"]
	print(f"\n================ STARTING TRAINING FOR {format_name} ==================")
	
	# 1. Load and process datasets
	print(f"Loading dataset from {config['train_path']}...")
	dataset = load_dataset("text", data_files={"train": config["train_path"], "val": config["val_path"]})
	processed_dataset = dataset.map(lambda x: apply_chat_template(x, tokenizer))
	
	# 2. Load model from base MODEL_ID
	print(f"Loading base model {MODEL_ID}...")
	
	if USE_UNSLOTH:
		model, tokenizer = FastLanguageModel.from_pretrained(
			model_name=MODEL_ID,
			max_seq_length=max_seq_length,
			dtype=COMPUTE_DTYPE,
			load_in_4bit=True,
		)
		# Unsloth uses its highly optimized PEFT wrapper
		model = FastLanguageModel.get_peft_model(
			model,
			r=16,
			target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
			lora_alpha=32,
			lora_dropout=0,
			bias="none",
			random_state=3407,
		)
	else:
		model = AutoModelForCausalLM.from_pretrained(
			MODEL_ID,
			quantization_config=bnb_config,
			device_map="auto",
			trust_remote_code=True,
			dtype=COMPUTE_DTYPE,
		)
		model = prepare_model_for_kbit_training(model)
		model = get_peft_model(model, peft_config)
		
	model.print_trainable_parameters()
 
	# print(f"Loading base model {MODEL_ID}...")
	# model = AutoModelForCausalLM.from_pretrained(
	# 	MODEL_ID,
	# 	quantization_config=bnb_config,
	# 	device_map="auto",
	# 	trust_remote_code=True,
	# 	dtype=COMPUTE_DTYPE,
	# )
	
	# model = prepare_model_for_kbit_training(model)
	# model = get_peft_model(model, peft_config)
	# model.print_trainable_parameters()
	
	# 3. Setup SFT Configuration
	if USE_UNSLOTH:
		from unsloth import is_bfloat16_supported
		use_bf16 = is_bfloat16_supported()
		use_fp16 = not use_bf16
	else:
		use_bf16 = True
		use_fp16 = False
	
	training_args = SFTConfig(
		output_dir=config["output_dir"],
		per_device_train_batch_size=1,
  		per_device_eval_batch_size=2,
		gradient_accumulation_steps=4,
		learning_rate=2e-4,
		logging_steps=10,
		logging_dir=f"{config['output_dir']}/logs",
		num_train_epochs=1,
		eval_strategy="steps",
		eval_steps=100,
		save_strategy="steps",
		save_steps=100,
		load_best_model_at_end=True,     # UPDATE INCASE OF OVERTRAINING
		metric_for_best_model="eval_loss", # UPDATE INCASE OF OVERTRAINING
		bf16=use_bf16,
		fp16=use_fp16,
		optim="paged_adamw_8bit",
		dataset_text_field="text",
		max_length=max_seq_length , # Used to be 1500...
	)
	
	# 4. Initialize SFTTrainer
	trainer = SFTTrainer(
		model=model,
		train_dataset=processed_dataset['train'],
		eval_dataset=processed_dataset['val'],
		processing_class=tokenizer,
		args=training_args,
		callbacks=[EarlyStoppingCallback(early_stopping_patience=3)], # UPDATE INCASE OF OVERTRAINING
	)
	
	# 5. Train and save model
	last_checkpoint = get_last_checkpoint(config["output_dir"])
	if last_checkpoint is not None:
		print(f"Resuming training from {last_checkpoint}...")
		trainer.train(resume_from_checkpoint=last_checkpoint)
	else:
		trainer.train()
	trainer.save_model(config["output_dir"])
		
	# 6. Memory Cleanup
	del trainer
	del model
	gc.collect()
	torch.cuda.empty_cache()
	print(f"🧹 Cleared GPU cache/VRAM for {format_name}.")
	print("=================================================================\n")


================ STARTING TRAINING FOR structOnly ==================
Loading dataset from ..//output/cache/train_structural.jsonl...


Loading base model mistralai/Mistral-Nemo-Instruct-2407...


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


...

Tokenizing eval dataset:   0%|          | 0/2039 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2039 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.318821,0.322343,0.325805,581905.000000,0.913971
200,0.260846,0.247437,0.245913,1162241.000000,0.934150
300,0.215183,0.205771,0.211188,1743336.000000,0.945251
400,0.192378,0.183393,0.195689,2327943.000000,0.951872
500,0.157661,0.167838,0.174551,2910762.000000,0.955865
600,0.162104,0.156544,0.164163,3494270.000000,0.959087
700,0.153811,0.148932,0.157320,4078241.000000,0.960981
800,0.146927,0.143532,0.145262,4657396.000000,0.962341
900,0.136640,0.138715,0.147614,5236250.000000,0.963583
1000,0.130744,0.135310,0.138057,5816386.000000,0.964422


🧹 Cleared GPU cache/VRAM for fullInfo.

